# Modelo Isolation Forest

En este notebook se desarrolla Isolation Forest como enfoque no supervisado para la detección de anomalías en el dataset LITNET-2020, manteniendo el protocolo experimental utilizado previamente para Regresión Logística y XGBoost.

El análisis principal corresponde a una tarea binaria entre tráfico normal (`attack_a = 0`) y tráfico de ataque (`attack_a = 1`). Sin embargo, a diferencia de los modelos supervisados, las etiquetas no serán utilizadas durante el ajuste del modelo principal.

Se consideran dos formulaciones experimentales:

1. **IF-U (Unsupervised)**: Isolation Forest se ajusta utilizando el conjunto TRAIN completo sin proporcionar etiquetas al algoritmo. Este constituye el experimento principal no supervisado.
2. **IF-N (Normal reference)**: Isolation Forest se ajusta únicamente utilizando tráfico normal de TRAIN. Este escenario se considera un análisis complementario de sensibilidad basado en una referencia limpia de normalidad.

Ambas formulaciones utilizan la partición congelada almacenada en `dataset_modelado_con_split.parquet`.

No se aplican técnicas supervisadas de tratamiento del desbalance como class weights, SMOTE, ADASYN o SMOTE+ENN.

In [1]:
from pathlib import Path
import sys
import json
import gc
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.ensemble import IsolationForest
RANDOM_STATE = 42

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data"
INTERIM_PATH = DATA_PATH / "interim"
RESULTS_PATH = PROJECT_ROOT / "results"
PREPROCESSING_RESULTS_PATH = RESULTS_PATH / "preprocessing"
IF_RESULTS_PATH = RESULTS_PATH / "modeling" / "isolation_forest"
IF_U_RESULTS_PATH = IF_RESULTS_PATH / "if_u"
IF_N_RESULTS_PATH = IF_RESULTS_PATH / "if_n"
MODELS_PATH = PROJECT_ROOT / "models"
IF_MODELS_PATH = MODELS_PATH / "isolation_forest"
MODELING_DATASET_PATH = INTERIM_PATH / "dataset_modelado_con_split.parquet"
for path in [IF_RESULTS_PATH, IF_U_RESULTS_PATH, IF_N_RESULTS_PATH, IF_MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

if not MODELING_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset oficial de modelado: {MODELING_DATASET_PATH}"
    )
print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print("\nDataset oficial de modelado:")
print(MODELING_DATASET_PATH)
print("\nResultados Isolation Forest:")
print(IF_RESULTS_PATH)
print("\nModelos Isolation Forest:")
print(IF_MODELS_PATH)

PROJECT_ROOT:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026

Dataset oficial de modelado:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\data\interim\dataset_modelado_con_split.parquet

Resultados Isolation Forest:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\isolation_forest

Modelos Isolation Forest:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\models\isolation_forest


In [3]:
# Carga de utilidades comunes
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from src.modeling_utils import (
    build_base_preprocessor,
    get_transformed_feature_names,
    load_split_dataframe,
    split_features_target,
    validate_imbalance_strategy_for_model,
    summarize_class_distribution,
)
print("Utilidades de modelado cargadas correctamente.")

Utilidades de modelado cargadas correctamente.


Carga de configuraciones

In [5]:
PREPROCESSING_CONFIG_PATH = PREPROCESSING_RESULTS_PATH / "preprocessing_config.json"
IMBALANCE_CONFIG_PATH = PREPROCESSING_RESULTS_PATH / "imbalance_config.json"
for config_path in [PREPROCESSING_CONFIG_PATH, IMBALANCE_CONFIG_PATH]:
    if not config_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {config_path}")

with open(PREPROCESSING_CONFIG_PATH, "r", encoding="utf-8") as file:
    preprocessing_config = json.load(file)
with open(IMBALANCE_CONFIG_PATH, "r", encoding="utf-8") as file:
    imbalance_config = json.load(file)
print("Configuraciones cargadas correctamente.")
print("\nNúmero de variables predictoras:",
      len(preprocessing_config["predictor_candidate_columns"]))
print("\nEstrategias registradas en imbalance_config:")
print(imbalance_config["imbalance_strategies"])

Configuraciones cargadas correctamente.

Número de variables predictoras: 22

Estrategias registradas en imbalance_config:
['none', 'class_weight', 'smote', 'adasyn', 'smote_enn']


In [6]:
# Verificación explícita de los 22 predictores
EXPECTED_PREDICTORS = [
    "td", "sa", "da", "sp", "dp", "pr",
    "tcp_flag_fin", "tcp_flag_syn", "tcp_flag_rst", "tcp_flag_psh",
    "tcp_flag_ack", "tcp_flag_urg", "tcp_flag_ece", "tcp_flag_cwr",
    "stos", "ipkt", "ibyt", "in_if", "out_if", "sas", "das", "exp"
]
predictor_columns = preprocessing_config["predictor_candidate_columns"]
if predictor_columns != EXPECTED_PREDICTORS:
    raise ValueError(
        "Las variables del archivo de configuración no coinciden con "
        "las 22 variables congeladas para modelado."
    )
print("Predictores verificados correctamente:")
print(len(predictor_columns), "variables")

Predictores verificados correctamente:
22 variables


Verificación del preprocesamiento definido para Isolation Forest

In [7]:
if_preprocessor_template = build_base_preprocessor(preprocessing_config=preprocessing_config, model_family="isolation_forest")
num_log_pipeline = next(
    transformer
    for name, transformer, _ in if_preprocessor_template.transformers
    if name == "num_log"
)
print("Preprocesador Isolation Forest reconocido correctamente.")
print("Pasos aplicados a td, ipkt e ibyt:", [name for name, _ in num_log_pipeline.steps])

Preprocesador Isolation Forest reconocido correctamente.
Pasos aplicados a td, ipkt e ibyt: ['imputer', 'log1p', 'scaler']


In [8]:
validate_imbalance_strategy_for_model(strategy_name="none", model_family="isolation_forest", imbalance_config=imbalance_config)
print("Estrategia verificada: Isolation Forest se entrenará sin técnicas supervisadas de desbalance.")

Estrategia verificada: Isolation Forest se entrenará sin técnicas supervisadas de desbalance.


Orden de ataques

-aca cambiarlo y preguntar porque el orden es distinto

In [9]:
ATTACK_CLASS_ORDER = ["icmp_f", "icmp_smf", "udp_f", "http_f", "tcp_w32_w", "tcp_red_w", "udp_reaper_w"]
IF_U_NAME = "if_u"
IF_N_NAME = "if_n"

In [10]:
# Bloqueo de Test
TEST_UNLOCKED = False
def load_if_split(split_name, columns=None):
    split_name = split_name.lower()
    if split_name == "test" and not TEST_UNLOCKED:
        raise RuntimeError("TEST permanece bloqueado. Debe habilitarse únicamente después de congelar todas las decisiones de Isolation Forest.")
    return load_split_dataframe(
        split_name=split_name,
        preprocessing_config=preprocessing_config,
        use_dev_sample=False,
        columns=columns,
        project_root=PROJECT_ROOT,
    )

## 1. Carga del dataset completo

En esta etapa se cargan TRAIN y VALID. TEST permanece bloqueado.

Para el experimento principal IF-U, `attack_a` no será proporcionada al algoritmo durante
el entrenamiento; se conserva únicamente para evaluar posteriormente la capacidad del
anomaly score para distinguir tráfico normal y de ataque.

Para el análisis complementario IF-N, `attack_a` será utilizada solamente para identificar
el subconjunto de tráfico normal de TRAIN utilizado como referencia.

La variable `attack_t` se carga únicamente en VALID para permitir posteriormente el análisis
de detección y distribución del anomaly score por cada tipo de ataque.

In [11]:
# Carga de TRAIN: 22 predictores + attack_a.
TRAIN_COLUMNS = predictor_columns + ["attack_a"]
df_train_full = load_if_split(split_name="train", columns=TRAIN_COLUMNS)
print("FULL TRAIN:", df_train_full.shape)

FULL TRAIN: (15803376, 23)


In [12]:
# Carga de VALID: 22 predictores + attack_a + attack_t.
VALID_COLUMNS = predictor_columns + ["attack_a", "attack_t"]
df_valid_full = load_if_split(split_name="valid", columns=VALID_COLUMNS)
print("FULL VALID:", df_valid_full.shape)

FULL VALID: (3386438, 24)


Separación entre predictores y etiqueta de evaluación

In [13]:
attack_t_valid = df_valid_full["attack_t"].astype("category").copy()

In [14]:
X_train_bin, y_train_bin = split_features_target(df=df_train_full, preprocessing_config=preprocessing_config, task="binary")
X_valid_bin, y_valid_bin = split_features_target(df=df_valid_full, preprocessing_config=preprocessing_config, task="binary")
print("TRAIN:")
print("X:", X_train_bin.shape)
print("y:", y_train_bin.shape)
print("\nVALID:")
print("X:", X_valid_bin.shape)
print("y:", y_valid_bin.shape)

TRAIN:
X: (15803376, 22)
y: (15803376,)

VALID:
X: (3386438, 22)
y: (3386438,)


In [15]:
del df_train_full, df_valid_full
gc.collect()

0

Verificación de distribución binaria

In [16]:
distribution_train = summarize_class_distribution(y_train_bin, name="train")
distribution_valid = summarize_class_distribution(y_valid_bin, name="valid")
class_distribution_binary = pd.concat([distribution_train, distribution_valid], ignore_index=True)
class_distribution_binary

,dataset,clase,n,porcentaje
0,train,0,14775206,93.493985
1,train,1,1028170,6.506015
2,valid,0,3166116,93.493990
3,valid,1,220322,6.506010


In [17]:
# Comprobación automática del split congelado
EXPECTED_TRAIN_COUNTS = {0: 14_775_206, 1: 1_028_170}
EXPECTED_VALID_COUNTS = {0: 3_166_116, 1: 220_322}
train_counts = y_train_bin.value_counts().sort_index().to_dict()
valid_counts = y_valid_bin.value_counts().sort_index().to_dict()
assert train_counts == EXPECTED_TRAIN_COUNTS, (f"Distribución TRAIN inesperada: {train_counts}")
assert valid_counts == EXPECTED_VALID_COUNTS, (f"Distribución VALID inesperada: {valid_counts}")
assert X_train_bin.shape == (15_803_376, 22)
assert X_valid_bin.shape == (3_386_438, 22)
print("Split congelado verificado correctamente.")
print("TRAIN:", train_counts)
print("VALID:", valid_counts)
print("TEST continúa bloqueado:", not TEST_UNLOCKED)

Split congelado verificado correctamente.
TRAIN: {0: 14775206, 1: 1028170}
VALID: {0: 3166116, 1: 220322}
TEST continúa bloqueado: True


In [18]:
# comprobación de la distribución de ataques en VALID
valid_attack_distribution = (attack_t_valid[y_valid_bin.eq(1)].value_counts().rename_axis("attack_t").reset_index(name="n"))
valid_attack_distribution

,attack_t,n
0,tcp_red_w,188355
1,udp_f,14037
2,icmp_smf,8922
3,tcp_w32_w,3644
4,http_f,3444
5,icmp_f,1744
6,udp_reaper_w,176
7,none,0


## 2. Preprocesamiento del escenario IF-U

El preprocesamiento se construye mediante `build_base_preprocessor()` utilizando
`model_family="isolation_forest"`. En esta configuración, las variables `td`, `ipkt` e
`ibyt` son imputadas mediante la mediana, transformadas con `log1p` y posteriormente
escaladas mediante `StandardScaler`. Las demás variables conservan las estrategias definidas
previamente: indicadores binarios, One-Hot Encoding, Frequency Encoding, características
derivadas de direcciones IP e indicadores asociados con rangos de puertos.

Debido a que los dos escenarios de Isolation Forest utilizan poblaciones de entrenamiento
diferentes, cada uno requiere un preprocesador independiente:

- **IF-U:** el preprocesador se ajusta utilizando TRAIN completo, sin emplear las etiquetas
  como información para las transformaciones.
- **IF-N:** el preprocesador se ajustará posteriormente utilizando únicamente el tráfico normal
  de TRAIN.


In [24]:
from scipy import sparse
def matrix_memory_gb(X):
    if sparse.issparse(X):
        n_bytes = X.data.nbytes + X.indices.nbytes + X.indptr.nbytes
    else:
        n_bytes = X.nbytes
    return n_bytes / (1024 ** 3)

In [25]:
if_u_preprocessor = build_base_preprocessor(preprocessing_config=preprocessing_config, model_family="isolation_forest")
print("Preprocesador IF-U construido correctamente.")

Preprocesador IF-U construido correctamente.


In [26]:
X_train_if_u = if_u_preprocessor.fit_transform(X_train_bin)
print("TRAIN IF-U transformado:", X_train_if_u.shape)

TRAIN IF-U transformado: (15803376, 49)


In [27]:
X_valid_if_u = if_u_preprocessor.transform(X_valid_bin)
print("VALID IF-U transformado:", X_valid_if_u.shape)

VALID IF-U transformado: (3386438, 49)


Verificación de la representación IF-U

In [28]:
print("TRAIN IF-U")
print("Shape:", X_train_if_u.shape)
print("Tipo:", type(X_train_if_u))
print("Dtype:", X_train_if_u.dtype)
print("Sparse:", sparse.issparse(X_train_if_u))
print("Memoria:", f"{matrix_memory_gb(X_train_if_u):.2f} GB")
print("\nVALID IF-U")
print("Shape:", X_valid_if_u.shape)
print("Tipo:", type(X_valid_if_u))
print("Dtype:", X_valid_if_u.dtype)
print("Sparse:", sparse.issparse(X_valid_if_u))
print("Memoria:", f"{matrix_memory_gb(X_valid_if_u):.2f} GB")

TRAIN IF-U
Shape: (15803376, 49)
Tipo: <class 'numpy.ndarray'>
Dtype: float32
Sparse: False
Memoria: 2.88 GB

VALID IF-U
Shape: (3386438, 49)
Tipo: <class 'numpy.ndarray'>
Dtype: float32
Sparse: False
Memoria: 0.62 GB


In [29]:
# Recuperación de nombres de características
if_u_feature_names = get_transformed_feature_names(if_u_preprocessor)
if if_u_feature_names is None:
    raise RuntimeError("No fue posible recuperar los nombres de las características transformadas.")
if len(if_u_feature_names) != X_train_if_u.shape[1]:
    raise ValueError("El número de nombres de características no coincide con la matriz transformada.")
print("Número de características transformadas:", len(if_u_feature_names))
print("\nPrimeras 15 características:")
for feature in if_u_feature_names[:15]:
    print(feature)

Número de características transformadas: 49

Primeras 15 características:
num_log__td
num_log__ipkt
num_log__ibyt
flags__tcp_flag_fin
flags__tcp_flag_syn
flags__tcp_flag_rst
flags__tcp_flag_psh
flags__tcp_flag_ack
flags__tcp_flag_urg
flags__tcp_flag_ece
flags__tcp_flag_cwr
onehot__pr_AH
onehot__pr_ESP
onehot__pr_GRE
onehot__pr_ICMP


In [33]:
num_log_pipeline = if_u_preprocessor.named_transformers_["num_log"]
print("Pipeline numérico IF-U:")
print([name for name, _ in num_log_pipeline.steps])
scaler_if_u = num_log_pipeline.named_steps["scaler"]
print("\nMedias aprendidas por StandardScaler:")
print(scaler_if_u.mean_)
print("\nEscalas aprendidas por StandardScaler:")
print(scaler_if_u.scale_)

Pipeline numérico IF-U:
['imputer', 'log1p', 'scaler']

Medias aprendidas por StandardScaler:
[0.65603582 1.95332175 6.13220599]

Escalas aprendidas por StandardScaler:
[1.23082691 1.00863839 1.68351682]


In [35]:
IF_U_FEATURES_PATH = IF_U_RESULTS_PATH / "if_u_transformed_features.csv"
pd.DataFrame({"feature": if_u_feature_names}).to_csv(IF_U_FEATURES_PATH, index=False)
IF_U_PREPROCESSOR_PATH = IF_MODELS_PATH / "if_u_preprocessor.joblib"
joblib.dump(if_u_preprocessor, IF_U_PREPROCESSOR_PATH)
print("Características IF-U guardadas en:")
print(IF_U_FEATURES_PATH)
print("\nPreprocesador IF-U guardado en:")
print(IF_U_PREPROCESSOR_PATH)

Características IF-U guardadas en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\isolation_forest\if_u\if_u_transformed_features.csv

Preprocesador IF-U guardado en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\models\isolation_forest\if_u_preprocessor.joblib


## 3. Isolation Forest no supervisado (IF-U)

Isolation Forest se evalúa inicialmente mediante el escenario IF-U, correspondiente al
experimento principal no supervisado del trabajo.

El modelo se ajusta utilizando la totalidad del conjunto TRAIN transformado, sin proporcionar
`attack_a` ni `attack_t` durante el entrenamiento. Por tanto, el algoritmo desconoce qué
observaciones corresponden a tráfico normal o de ataque y construye los árboles únicamente
a partir de la distribución de las características.
Como configuración baseline se utiliza una parametrización cercana a la propuesta original
de Isolation Forest y a los valores por defecto de scikit-learn:

- 100 árboles.
- submuestras de 256 observaciones por árbol mediante `max_samples="auto"`.
- todas las características disponibles (`max_features=1.0`).
- muestreo sin reemplazo (`bootstrap=False`).
- `contamination="auto"`.
- `random_state=42`.

El objetivo del baseline es establecer primero la capacidad natural de Isolation Forest antes
de modificar su estructura.

In [37]:
del X_train_bin, X_valid_bin
# El preprocesador IF-U ya fue guardado en joblib. Se podrá recargar posteriormente para TEST.
del if_u_preprocessor
if "if_preprocessor_template" in globals():
    del if_preprocessor_template
gc.collect()
print("Objetos originales liberados.")
print("TRAIN IF-U disponible:", X_train_if_u.shape)
print("VALID IF-U disponible:", X_valid_if_u.shape)
print("TEST desbloqueado:", TEST_UNLOCKED)

Objetos originales liberados.
TRAIN IF-U disponible: (15803376, 49)
VALID IF-U disponible: (3386438, 49)
TEST desbloqueado: False


In [39]:
print("scikit-learn:", sklearn.__version__)
IF_U_BASE_PARAMS = {
    "n_estimators": 100,
    "max_samples": "auto",
    "contamination": "auto",
    "max_features": 1.0,
    "bootstrap": False,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}
if_u_baseline = IsolationForest(**IF_U_BASE_PARAMS)
print("\nConfiguración baseline IF-U:")
for key, value in IF_U_BASE_PARAMS.items():
    print(f"{key}: {value}")

scikit-learn: 1.7.1

Configuración baseline IF-U:
n_estimators: 100
max_samples: auto
contamination: auto
max_features: 1.0
bootstrap: False
n_jobs: -1
random_state: 42


### 3.1 Entrenamiento del baseline sobre entrenamiento completo

In [40]:
import time

In [41]:
start_time = time.perf_counter()
if_u_baseline.fit(X_train_if_u)
if_u_training_seconds = time.perf_counter() - start_time
print("IF-U baseline entrenado correctamente.")
print(f"Tiempo de entrenamiento: {if_u_training_seconds:.2f} s "
      f"({if_u_training_seconds / 60:.2f} min)")
print("Número de árboles:", len(if_u_baseline.estimators_))
print("max_samples efectivo:", if_u_baseline.max_samples_)
print("offset_:", if_u_baseline.offset_)

IF-U baseline entrenado correctamente.
Tiempo de entrenamiento: 8.22 s (0.14 min)
Número de árboles: 100
max_samples efectivo: 256
offset_: -0.5


In [42]:
train_attack_prevalence = float(np.mean(y_train_bin))
expected_attacks_per_tree = if_u_baseline.max_samples_ * train_attack_prevalence
print("\nDiagnóstico de la población IF-U:")
print(f"Prevalencia de ataque en TRAIN: {train_attack_prevalence:.6f}")
print(f"Ataques esperados por submuestra de {if_u_baseline.max_samples_}: "
      f"{expected_attacks_per_tree:.2f}")


Diagnóstico de la población IF-U:
Prevalencia de ataque en TRAIN: 0.065060
Ataques esperados por submuestra de 256: 16.66


Es importante entender que este cálculo usa las etiquetas después del entrenamiento únicamente para describir el experimento. No fueron proporcionadas al algoritmo.

In [43]:
IF_U_BASELINE_MODEL_PATH = IF_MODELS_PATH / "if_u_baseline.joblib"
joblib.dump(if_u_baseline, IF_U_BASELINE_MODEL_PATH)
print("Modelo IF-U baseline guardado en:")
print(IF_U_BASELINE_MODEL_PATH)

Modelo IF-U baseline guardado en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\models\isolation_forest\if_u_baseline.joblib


In [44]:
del X_train_if_u
gc.collect()
print("Matriz TRAIN IF-U liberada después del entrenamiento.")
print("VALID IF-U permanece disponible:", X_valid_if_u.shape)

Matriz TRAIN IF-U liberada después del entrenamiento.
VALID IF-U permanece disponible: (3386438, 49)


### 3.2 Obtención del anomaly score sobre validación

In [45]:
start_time = time.perf_counter()
with joblib.parallel_backend("threading", n_jobs=-1):
    raw_score_valid_if_u = if_u_baseline.score_samples(X_valid_if_u)
if_u_scoring_seconds = time.perf_counter() - start_time
anomaly_score_valid_if_u = -raw_score_valid_if_u
print("Scores de VALID calculados correctamente.")
print(f"Tiempo de scoring: {if_u_scoring_seconds:.2f} s "
      f"({if_u_scoring_seconds / 60:.2f} min)")

print("\nscore_samples original:")
print("Min:", raw_score_valid_if_u.min())
print("Max:", raw_score_valid_if_u.max())

print("\nAnomaly score orientado a ataque:")
print("Min:", anomaly_score_valid_if_u.min())
print("Max:", anomaly_score_valid_if_u.max())

Scores de VALID calculados correctamente.
Tiempo de scoring: 7.40 s (0.12 min)

score_samples original:
Min: -0.6628161751206164
Max: -0.4033270737044173

Anomaly score orientado a ataque:
Min: 0.4033270737044173
Max: 0.6628161751206164


### 3.3 Punto operativo automático de Isolation Forest

In [46]:
auto_threshold_if_u = -float(if_u_baseline.offset_)
# Equivalente exacto a decision_function < 0
decision_valid_if_u = raw_score_valid_if_u - if_u_baseline.offset_
y_pred_auto_if_u = (decision_valid_if_u < 0).astype(np.uint8)
# Misma regla expresada utilizando nuestro anomaly score
y_pred_score_if_u = (anomaly_score_valid_if_u > auto_threshold_if_u).astype(np.uint8)
assert np.array_equal(y_pred_auto_if_u, y_pred_score_if_u)
print("Threshold automático en anomaly_score:", auto_threshold_if_u)
print("Predicciones equivalentes verificadas correctamente.")
print("Alertas generadas:", f"{y_pred_auto_if_u.sum():,}")
print("Proporción de VALID marcada como anomalía:",
      f"{y_pred_auto_if_u.mean():.4%}")

Threshold automático en anomaly_score: 0.5
Predicciones equivalentes verificadas correctamente.
Alertas generadas: 714,013
Proporción de VALID marcada como anomalía: 21.0845%


### 3.4 Función de evaluación comparable con LR y XGBoost

In [47]:
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve, auc)
def evaluate_binary_scores(y_true, y_score, threshold, experiment_name="experiment", y_pred=None):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if y_pred is None:
        y_pred = (y_score > threshold).astype(np.uint8)
    else:
        y_pred = np.asarray(y_pred).astype(np.uint8)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    fnr = fn / (fn + tp) if (fn + tp) else np.nan
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall_curve, precision_curve)
    return {
        "experiment": experiment_name,
        "threshold": threshold,
        "n": len(y_true),
        "attack_prevalence": y_true.mean(),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "alerts": int(tp + fp),
        "alert_rate": (tp + fp) / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "FPR": fpr,
        "FNR": fnr,
        "ROC_AUC": roc_auc_score(y_true, y_score),
        "PR_AUC": pr_auc,
    }

In [48]:
metrics_if_u_auto = evaluate_binary_scores(y_true=y_valid_bin, y_score=anomaly_score_valid_if_u, threshold=auto_threshold_if_u, experiment_name="IF-U baseline auto", y_pred=y_pred_auto_if_u)
results_if_u_auto = pd.DataFrame([metrics_if_u_auto])
display(results_if_u_auto[["experiment", "threshold", "precision", "recall", "f1", "PR_AUC", "ROC_AUC", "balanced_accuracy", "specificity", "FPR", "FNR", "TN", "FP", "FN", "TP", "alerts", "alert_rate"]].style.format({
        "threshold": "{:.6f}",
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "f1": "{:.6f}",
        "PR_AUC": "{:.6f}",
        "ROC_AUC": "{:.6f}",
        "balanced_accuracy": "{:.6f}",
        "specificity": "{:.6f}",
        "FPR": "{:.6f}",
        "FNR": "{:.6f}",
        "alert_rate": "{:.4%}",
        "TN": "{:,.0f}",
        "FP": "{:,.0f}",
        "FN": "{:,.0f}",
        "TP": "{:,.0f}",
        "alerts": "{:,.0f}",
    })
)

,experiment,threshold,precision,recall,f1,PR_AUC,ROC_AUC,balanced_accuracy,specificity,FPR,FNR,TN,FP,FN,TP,alerts,alert_rate
0,IF-U baseline auto,0.500000,0.032980,0.106880,0.050406,0.056916,0.263888,0.444400,0.781920,0.218080,0.893120,"2,475,651","690,465","196,774","23,548","714,013",21.0845%


In [49]:
row = results_if_u_auto.iloc[0]
print("ISOLATION FOREST IF-U BASELINE — VALID")
print("-" * 50)
print(f"Precision ataque:   {row['precision']:.6f}")
print(f"Recall ataque:      {row['recall']:.6f}")
print(f"F1 ataque:          {row['f1']:.6f}")
print(f"PR-AUC:             {row['PR_AUC']:.6f}")
print(f"ROC-AUC:            {row['ROC_AUC']:.6f}")
print(f"Balanced accuracy:  {row['balanced_accuracy']:.6f}")
print(f"Specificity:        {row['specificity']:.6f}")
print()
print(f"FPR:                {row['FPR']:.6f} ({row['FPR']:.4%})")
print(f"FNR:                {row['FNR']:.6f} ({row['FNR']:.4%})")
print()
print(f"TN: {int(row['TN']):,}")
print(f"FP: {int(row['FP']):,}")
print(f"FN: {int(row['FN']):,}")
print(f"TP: {int(row['TP']):,}")
print()
print(f"Alertas:            {int(row['alerts']):,}")
print(f"Tasa de alertas:    {row['alert_rate']:.4%}")

ISOLATION FOREST IF-U BASELINE — VALID
--------------------------------------------------
Precision ataque:   0.032980
Recall ataque:      0.106880
F1 ataque:          0.050406
PR-AUC:             0.056916
ROC-AUC:            0.263888
Balanced accuracy:  0.444400
Specificity:        0.781920

FPR:                0.218080 (21.8080%)
FNR:                0.893120 (89.3120%)

TN: 2,475,651
FP: 690,465
FN: 196,774
TP: 23,548

Alertas:            714,013
Tasa de alertas:    21.0845%


**¿El score realmente separa normal y ataque?**

In [50]:
def summarize_binary_scores(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    rows = []
    for label, name in [(0, "normal"), (1, "attack")]:
        values = y_score[y_true == label]
        rows.append({"clase": name, "n": len(values), "mean": values.mean(), "std": values.std(), "p05": np.quantile(values, 0.05), "p25": np.quantile(values, 0.25), "median": np.median(values), "p75": np.quantile(values, 0.75), "p95": np.quantile(values, 0.95)})
    return pd.DataFrame(rows)

score_summary_if_u = summarize_binary_scores(y_valid_bin, anomaly_score_valid_if_u)
display(score_summary_if_u.style.format({"n": "{:,.0f}", "mean": "{:.6f}", "std": "{:.6f}", "p05": "{:.6f}", "p25": "{:.6f}", "median": "{:.6f}", "p75": "{:.6f}", "p95": "{:.6f}"}))

,clase,n,mean,std,p05,p25,median,p75,p95
0,normal,"3,166,116",0.475925,0.034202,0.425381,0.451345,0.473018,0.496472,0.538265
1,attack,"220,322",0.450238,0.046268,0.405525,0.414780,0.439751,0.464115,0.561801


**Recall por tipo de ataque**

In [51]:
def evaluate_recall_by_attack_if(y_true, y_score, attack_t, threshold, attack_order=None):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    attack_t = np.asarray(attack_t)
    attack_mask = (y_true == 1)
    df = pd.DataFrame({"attack_t": attack_t[attack_mask].astype(str), "anomaly_score": y_score[attack_mask]})
    df["detected"] = (df["anomaly_score"] > threshold).astype(np.uint8)
    results = (df.groupby("attack_t", as_index=False).agg(n_attacks=("detected", "size"), detected=("detected", "sum"), mean_score=("anomaly_score", "mean"), median_score=("anomaly_score", "median")))
    results["fn"] = results["n_attacks"] - results["detected"]
    results["recall"] = results["detected"] / results["n_attacks"]
    if attack_order is not None:
        results["attack_t"] = pd.Categorical(results["attack_t"], categories=attack_order, ordered=True)
        results = (results.sort_values("attack_t").reset_index(drop=True))
    macro_recall = results["recall"].mean()
    return results, macro_recall

In [52]:
recall_if_u_auto, macro_recall_if_u_auto = (evaluate_recall_by_attack_if(y_true=y_valid_bin, y_score=anomaly_score_valid_if_u, attack_t=attack_t_valid, threshold=auto_threshold_if_u, attack_order=ATTACK_CLASS_ORDER))
display(recall_if_u_auto.style.format({"n_attacks": "{:,.0f}", "detected": "{:,.0f}", "mean_score": "{:.6f}", "median_score": "{:.6f}", "fn": "{:,.0f}", "recall": "{:.6f}"}))
print("Macro-recall IF-U baseline:", f"{macro_recall_if_u_auto:.6f}")

,attack_t,n_attacks,detected,mean_score,median_score,fn,recall
0,icmp_f,"1,744","1,744",0.549895,0.545811,0,1.000000
1,icmp_smf,"8,922","8,922",0.563260,0.561120,0,1.000000
2,udp_f,"14,037","8,897",0.548616,0.577548,"5,140",0.633825
3,http_f,"3,444","1,757",0.500779,0.503601,"1,687",0.510163
4,tcp_w32_w,"3,644",248,0.448983,0.446080,"3,396",0.068057
5,tcp_red_w,"188,355","1,979",0.435720,0.436580,"186,376",0.010507
6,udp_reaper_w,176,1,0.461165,0.460080,175,0.005682


Macro-recall IF-U baseline: 0.461176


In [53]:
results_if_u_auto["macro_recall"] = macro_recall_if_u_auto
results_if_u_auto["training_seconds"] = if_u_training_seconds
results_if_u_auto["scoring_seconds"] = if_u_scoring_seconds
results_if_u_auto["max_samples_effective"] = if_u_baseline.max_samples_
results_if_u_auto["offset"] = if_u_baseline.offset_
IF_U_BASELINE_RESULTS_PATH = (IF_U_RESULTS_PATH / "if_u_baseline_valid_metrics.csv")
results_if_u_auto.to_csv(IF_U_BASELINE_RESULTS_PATH, index=False)
IF_U_RECALL_PATH = (IF_U_RESULTS_PATH / "if_u_baseline_recall_by_attack.csv")
recall_if_u_auto.to_csv(IF_U_RECALL_PATH, index=False)

In [54]:
valid_scores_if_u = pd.DataFrame({
    "attack_a": np.asarray(y_valid_bin, dtype=np.uint8),
    "attack_t": attack_t_valid.array,
    "anomaly_score": anomaly_score_valid_if_u,
    "pred_auto": y_pred_auto_if_u,
})
IF_U_VALID_SCORES_PATH = (IF_U_RESULTS_PATH / "if_u_baseline_valid_scores.parquet")
valid_scores_if_u.to_parquet(IF_U_VALID_SCORES_PATH, index=False)
print("Resultados guardados:")
print(IF_U_BASELINE_RESULTS_PATH)
print(IF_U_RECALL_PATH)
print(IF_U_VALID_SCORES_PATH)

Resultados guardados:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\isolation_forest\if_u\if_u_baseline_valid_metrics.csv
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\isolation_forest\if_u\if_u_baseline_recall_by_attack.csv
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\isolation_forest\if_u\if_u_baseline_valid_scores.parquet


In [55]:
del valid_scores_if_u
gc.collect()

20

## 4. Isolation Forest con referencia normal (IF-N)

Como escenario complementario se evalúa una segunda formulación de Isolation Forest
utilizando exclusivamente el tráfico normal de TRAIN como población de referencia.

La etiqueta `attack_a` se utiliza únicamente para seleccionar los registros normales
(`attack_a = 0`). Posteriormente, tanto el preprocesamiento como Isolation Forest son
ajustados sin proporcionar una variable objetivo al estimador.

A diferencia de IF-U, las estadísticas utilizadas en el escalamiento y las frecuencias
aprendidas durante las codificaciones corresponden únicamente al tráfico normal. VALID
permanece completo, incluyendo tráfico normal y los siete tipos de ataque, con el propósito
de evaluar qué tan atípicas resultan estas observaciones respecto de la referencia aprendida.

Para aislar el efecto de la población de entrenamiento se mantiene exactamente la misma
configuración estructural utilizada en el baseline IF-U.

In [56]:
if "X_valid_if_u" in globals():
    del X_valid_if_u
if "raw_score_valid_if_u" in globals():
    del raw_score_valid_if_u
gc.collect()
print("Matrices IF-U no necesarias liberadas.")
print("Resultados IF-U conservados para comparación.")
print("TEST desbloqueado:", TEST_UNLOCKED)

Matrices IF-U no necesarias liberadas.
Resultados IF-U conservados para comparación.
TEST desbloqueado: False


In [57]:
import duckdb
dataset_sql_path = (MODELING_DATASET_PATH.resolve().as_posix().replace("'", "''"))
predictor_sql = ", ".join(f'"{col}"' for col in predictor_columns)
query_normal_train = f"""
SELECT {predictor_sql}
FROM read_parquet('{dataset_sql_path}')
WHERE split = 'train'
  AND attack_a = 0
"""
connection = duckdb.connect()
try:
    X_train_if_n_raw = connection.execute(query_normal_train).df()
finally:
    connection.close()
print("TRAIN normal IF-N:", X_train_if_n_raw.shape)

TRAIN normal IF-N: (14775206, 22)


In [58]:
X_valid_if_n_raw = load_if_split(split_name="valid", columns=predictor_columns)
print("VALID IF-N:", X_valid_if_n_raw.shape)

VALID IF-N: (3386438, 22)


Preprocesador independiente IF-N

In [60]:
if_n_preprocessor = build_base_preprocessor(preprocessing_config=preprocessing_config, model_family="isolation_forest")
print("Preprocesador IF-N construido correctamente.")

Preprocesador IF-N construido correctamente.


In [61]:
X_train_if_n = if_n_preprocessor.fit_transform(X_train_if_n_raw)
print("TRAIN IF-N transformado:", X_train_if_n.shape)

TRAIN IF-N transformado: (14775206, 49)


In [62]:
X_valid_if_n = if_n_preprocessor.transform(X_valid_if_n_raw)
print("VALID IF-N transformado:", X_valid_if_n.shape)

VALID IF-N transformado: (3386438, 49)


In [63]:
print("TRAIN IF-N")
print("Shape:", X_train_if_n.shape)
print("Tipo:", type(X_train_if_n))
print("Dtype:", X_train_if_n.dtype)
print("Sparse:", sparse.issparse(X_train_if_n))
print("Memoria:", f"{matrix_memory_gb(X_train_if_n):.2f} GB")
print("\nVALID IF-N")
print("Shape:", X_valid_if_n.shape)
print("Tipo:", type(X_valid_if_n))
print("Dtype:", X_valid_if_n.dtype)
print("Sparse:", sparse.issparse(X_valid_if_n))
print("Memoria:", f"{matrix_memory_gb(X_valid_if_n):.2f} GB")

TRAIN IF-N
Shape: (14775206, 49)
Tipo: <class 'numpy.ndarray'>
Dtype: float32
Sparse: False
Memoria: 2.70 GB

VALID IF-N
Shape: (3386438, 49)
Tipo: <class 'numpy.ndarray'>
Dtype: float32
Sparse: False
Memoria: 0.62 GB


In [64]:
if_n_feature_names = get_transformed_feature_names(if_n_preprocessor)
if if_n_feature_names is None:
    raise RuntimeError("No fue posible recuperar las características transformadas de IF-N.")
assert len(if_n_feature_names) == X_train_if_n.shape[1]
print("Número de características IF-N:", len(if_n_feature_names))

Número de características IF-N: 49


In [65]:
if "if_u_feature_names" in globals():
    only_if_u = sorted(set(if_u_feature_names) - set(if_n_feature_names))
    only_if_n = sorted(set(if_n_feature_names) - set(if_u_feature_names))
    print("\nCaracterísticas presentes en IF-U y ausentes en IF-N:")
    print(only_if_u if only_if_u else "Ninguna")
    print("\nCaracterísticas presentes en IF-N y ausentes en IF-U:")
    print(only_if_n if only_if_n else "Ninguna")


Características presentes en IF-U y ausentes en IF-N:
Ninguna

Características presentes en IF-N y ausentes en IF-U:
Ninguna


In [67]:
IF_N_FEATURES_PATH = IF_N_RESULTS_PATH / "if_n_transformed_features.csv"
pd.DataFrame({"feature": if_n_feature_names}).to_csv(IF_N_FEATURES_PATH, index=False)
IF_N_PREPROCESSOR_PATH = IF_MODELS_PATH / "if_n_preprocessor.joblib"
joblib.dump(if_n_preprocessor, IF_N_PREPROCESSOR_PATH)
print("Características IF-N guardadas en:")
print(IF_N_FEATURES_PATH)
print("\nPreprocesador IF-N guardado en:")
print(IF_N_PREPROCESSOR_PATH)

Características IF-N guardadas en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\isolation_forest\if_n\if_n_transformed_features.csv

Preprocesador IF-N guardado en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\models\isolation_forest\if_n_preprocessor.joblib


In [68]:
del X_train_if_n_raw, X_valid_if_n_raw
gc.collect()
print("DataFrames originales IF-N liberados.")
print("TRAIN transformado:", X_train_if_n.shape)
print("VALID transformado:", X_valid_if_n.shape)

DataFrames originales IF-N liberados.
TRAIN transformado: (14775206, 49)
VALID transformado: (3386438, 49)


Configuración baseline IF-N

In [69]:
IF_N_BASE_PARAMS = IF_U_BASE_PARAMS.copy()
print("Configuración baseline IF-N:")
for key, value in IF_N_BASE_PARAMS.items():
    print(f"{key}: {value}")

Configuración baseline IF-N:
n_estimators: 100
max_samples: auto
contamination: auto
max_features: 1.0
bootstrap: False
n_jobs: -1
random_state: 42


Entrenamiento IF-N

In [70]:
if_n_baseline = IsolationForest(**IF_N_BASE_PARAMS)
start_time = time.perf_counter()
if_n_baseline.fit(X_train_if_n)
if_n_training_seconds = time.perf_counter() - start_time
print("IF-N baseline entrenado correctamente.")
print(f"Tiempo de entrenamiento: {if_n_training_seconds:.2f} s " f"({if_n_training_seconds / 60:.2f} min)")
print("Número de árboles:", len(if_n_baseline.estimators_))
print("max_samples efectivo:", if_n_baseline.max_samples_)
print("offset_:", if_n_baseline.offset_)

IF-N baseline entrenado correctamente.
Tiempo de entrenamiento: 8.19 s (0.14 min)
Número de árboles: 100
max_samples efectivo: 256
offset_: -0.5


In [71]:
IF_N_BASELINE_MODEL_PATH = IF_MODELS_PATH / "if_n_baseline.joblib"
joblib.dump(if_n_baseline, IF_N_BASELINE_MODEL_PATH)
print("Modelo IF-N baseline guardado en:")
print(IF_N_BASELINE_MODEL_PATH)

Modelo IF-N baseline guardado en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\models\isolation_forest\if_n_baseline.joblib


In [72]:
del X_train_if_n
gc.collect()
print("TRAIN IF-N transformado liberado.")
print("VALID IF-N disponible:", X_valid_if_n.shape)

TRAIN IF-N transformado liberado.
VALID IF-N disponible: (3386438, 49)


Anomaly score sobre VALID

In [73]:
start_time = time.perf_counter()
with joblib.parallel_backend("threading", n_jobs=-1):
    raw_score_valid_if_n = if_n_baseline.score_samples(X_valid_if_n)
if_n_scoring_seconds = time.perf_counter() - start_time
anomaly_score_valid_if_n = -raw_score_valid_if_n
print("Scores IF-N calculados correctamente.")
print(f"Tiempo de scoring: {if_n_scoring_seconds:.2f} s " f"({if_n_scoring_seconds / 60:.2f} min)")
print("\nAnomaly score IF-N:")
print("Min:", anomaly_score_valid_if_n.min())
print("Max:", anomaly_score_valid_if_n.max())

Scores IF-N calculados correctamente.
Tiempo de scoring: 5.98 s (0.10 min)

Anomaly score IF-N:
Min: 0.4051022585763905
Max: 0.6702890589864603


Punto operativo automático

In [74]:
auto_threshold_if_n = -float(if_n_baseline.offset_)
decision_valid_if_n = (raw_score_valid_if_n - if_n_baseline.offset_)
y_pred_auto_if_n = (decision_valid_if_n < 0).astype(np.uint8)
y_pred_score_if_n = (anomaly_score_valid_if_n > auto_threshold_if_n).astype(np.uint8)
assert np.array_equal(y_pred_auto_if_n, y_pred_score_if_n)
print("Threshold automático IF-N:", auto_threshold_if_n)
print("Predicciones equivalentes verificadas.")
print("Alertas generadas:", f"{y_pred_auto_if_n.sum():,}")
print("Proporción de VALID marcada como anomalía:", f"{y_pred_auto_if_n.mean():.4%}")

Threshold automático IF-N: 0.5
Predicciones equivalentes verificadas.
Alertas generadas: 682,028
Proporción de VALID marcada como anomalía: 20.1400%


Métricas del baseline IF-N

In [75]:
metrics_if_n_auto = evaluate_binary_scores(
    y_true=y_valid_bin,
    y_score=anomaly_score_valid_if_n,
    threshold=auto_threshold_if_n,
    experiment_name="IF-N baseline auto",
    y_pred=y_pred_auto_if_n,
)
results_if_n_auto = pd.DataFrame([metrics_if_n_auto])

In [76]:
row = results_if_n_auto.iloc[0]
print("ISOLATION FOREST IF-N BASELINE — VALID")
print("-" * 50)
print(f"Precision ataque:   {row['precision']:.6f}")
print(f"Recall ataque:      {row['recall']:.6f}")
print(f"F1 ataque:          {row['f1']:.6f}")
print(f"PR-AUC:             {row['PR_AUC']:.6f}")
print(f"ROC-AUC:            {row['ROC_AUC']:.6f}")
print(f"Balanced accuracy:  {row['balanced_accuracy']:.6f}")
print(f"Specificity:        {row['specificity']:.6f}")
print()
print(f"FPR:                {row['FPR']:.6f} ({row['FPR']:.4%})")
print(f"FNR:                {row['FNR']:.6f} ({row['FNR']:.4%})")
print()
print(f"TN: {int(row['TN']):,}")
print(f"FP: {int(row['FP']):,}")
print(f"FN: {int(row['FN']):,}")
print(f"TP: {int(row['TP']):,}")
print()
print(f"Alertas:            {int(row['alerts']):,}")
print(f"Tasa de alertas:    {row['alert_rate']:.4%}")

ISOLATION FOREST IF-N BASELINE — VALID
--------------------------------------------------
Precision ataque:   0.041967
Recall ataque:      0.129914
F1 ataque:          0.063441
PR-AUC:             0.058786
ROC-AUC:            0.377111
Balanced accuracy:  0.461770
Specificity:        0.793626

FPR:                0.206374 (20.6374%)
FNR:                0.870086 (87.0086%)

TN: 2,512,711
FP: 653,405
FN: 191,699
TP: 28,623

Alertas:            682,028
Tasa de alertas:    20.1400%


Distribución de scores

In [77]:
score_summary_if_n = summarize_binary_scores(y_valid_bin, anomaly_score_valid_if_n)
display(score_summary_if_n.style.format({
        "n": "{:,.0f}",
        "mean": "{:.6f}",
        "std": "{:.6f}",
        "p05": "{:.6f}",
        "p25": "{:.6f}",
        "median": "{:.6f}",
        "p75": "{:.6f}",
        "p95": "{:.6f}",
    })
)

,clase,n,mean,std,p05,p25,median,p75,p95
0,normal,"3,166,116",0.474226,0.034579,0.426323,0.449021,0.470062,0.494936,0.539853
1,attack,"220,322",0.463085,0.040713,0.421006,0.428078,0.455209,0.481687,0.562653


In [78]:
recall_if_n_auto, macro_recall_if_n_auto = (
    evaluate_recall_by_attack_if(
        y_true=y_valid_bin,
        y_score=anomaly_score_valid_if_n,
        attack_t=attack_t_valid,
        threshold=auto_threshold_if_n,
        attack_order=ATTACK_CLASS_ORDER,
    )
)

display(
    recall_if_n_auto.style.format({
        "n_attacks": "{:,.0f}",
        "detected": "{:,.0f}",
        "mean_score": "{:.6f}",
        "median_score": "{:.6f}",
        "fn": "{:,.0f}",
        "recall": "{:.6f}",
    })
)

print(
    "Macro-recall IF-N baseline:",
    f"{macro_recall_if_n_auto:.6f}",
)

,attack_t,n_attacks,detected,mean_score,median_score,fn,recall
0,icmp_f,"1,744","1,744",0.552106,0.547194,0,1.000000
1,icmp_smf,"8,922","8,922",0.564449,0.562653,0,1.000000
2,udp_f,"14,037","7,627",0.535547,0.559040,"6,410",0.543350
3,http_f,"3,444",953,0.485332,0.492681,"2,491",0.276713
4,tcp_w32_w,"3,644",244,0.451048,0.443592,"3,400",0.066959
5,tcp_red_w,"188,355","9,133",0.451903,0.449145,"179,222",0.048488
6,udp_reaper_w,176,0,0.444195,0.439482,176,0.000000


Macro-recall IF-N baseline: 0.419359
